In [ ]:
import sys
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import signal
from pathlib import Path

### Create main df

In [ ]:


sys.path.append(os.path.abspath("/home/anushkasingh/Desktop/Thesis/Code/Baseline Correction"))
from load_data import read_data, create_combined_dataset
from baseline_correct import baseline_roy, process_all_samples
from best_params import fsvm_nested_cv, fsvm_nested_cv_XieOgden

path = ["../ALLDataGross/allKgData",
    "../ALLDataGross/BlindData",
    "../ALLDataGross/healthyCohort"
]
normVP = [[504, 425, 451, 454, 450, 474, 451, 471, 540, 467,
    550, 468, 481, 450, 515, 441, 452, 462, 453, 450, 452, 
    490, 504, 520, 525, 498, 542, 527, 550],
        [505, 503, 478, 453, 460, 494, 410, 413, 479, 489, 
    473, 464, 445, 499, 406, 455, 481, 388, 428, 466, 463, 
    520, 461],
    [420, 420, 428, 448, 417, 430, 420, 449, 483, 499, 
    438, 465, 438, 428, 503, 505, 504, 454, 515, 441, 
    404, 363]]
infoP = [["H", "PC", "PC", "H", "PC", "BC", "PC", "PC", "BC", "BC", 
    "PC", "PC", "PC", "PC", "H", "H","PC", "PC", "KC", "PC", "KC", 
    "PC", "BC", "BC", "PC", "KC", "PC", "PC", "PC"],
        ["H", "H" ,"H" ,"H", "KC", "KC", "PC", "PC" , "PC" ,"PC", 
        "PC", "PC", 'BC', 'KC', 'PC' , 'PC' , 'PC', 'KC' , 'PC', 'BC', 'PC', 'KC', 'BC'],
        ["F", "M", "M", "F", "F", "F", "F", "M", "M", "M", 
    "M", "F", "M", "M", "F", "M" ,"M", "M", "M", "M", 
    "M", "M" ]
]

df = create_combined_dataset(path,normVP,infoP)

path = "../ALLDataGross/healthyCohort"
dataframes = read_data(path)


### Make other relevant dfs

In [ ]:
                                                                                                                                                                                                        
# 1. wavenumber array — all rows should share the same wavenumbers, take the first                                                                                                                             
x = np.array(df['wavenumber'].iloc[0])  # shape (Mf,)                                                                                                                                                          
                                                                                                                                                                                                        
# 2. intensity matrix — stack each row's array as a column → shape (Mf, Nf)                                                                                                                                    
all_spectra = np.column_stack(df['intensity'].values)  # shape (Mf, Nf)                                                                                                                                        
                                                                                                                                                                                                        
# 3. norm factors — one per sample → shape (Nf,)                                                                                                                                                               
norm_factors = df['normVP'].values  # shape (Nf,)
                                                                                                                                                                                                        
# 4. call                                                                                                                                                                                                      
dataS, av_data, av_smoothed = process_all_samples(x, all_spectra, norm_factors)

### Perform Baseline Correction

In [ ]:
df['intensity_baseline_corrected'] = [dataS[:, i] for i in range(dataS.shape[1])]

### Update class

In [ ]:
df["infoP"].value_counts()
df["class"] = df["infoP"].apply(lambda x: x if x in ["PC", "KC", "BC", "H"] else "H")

In [ ]:
df["class"].value_counts()
df_whole = df

In [ ]:
df_whole.drop_duplicates(subset=["original_filename"], keep = "first", inplace = True)


### Plots for single sample spectra

In [ ]:
# Example usage
from matplotlib import style

i = 4
normVP = [420, 420, 428, 448, 417, 430, 420, 449, 483, 499, 
    438, 465, 438, 428, 503, 505, 504, 454, 515, 441, 
    404, 363]
filename, df = dataframes[i]
norm_factor_i = normVP[i]
x = df['Wavenumber'].values
y = df['Intensity'].values

segment_edges = [990, 1020]
y_stage3, y_stage2, y_stage1  = baseline_roy(x, y, norm_factor_i=norm_factor_i)
# y_stage2, y_stage1 = baseline_intermediates(x, y, norm_factor_i = norm_factor_i)
baseline = y - y_stage3

# visualize
plt.figure(figsize=(10,6))
plt.plot(x, y, label='Original', color='grey')
plt.plot(x, y_stage1, label='1st order (Shift)', color = 'blue')
plt.plot(x, y_stage2, label='2nd order Tilt removed)', color ='red')
# plt.plot(x, y_stage3, label='3rd order (Segment-corrected)')
# plt.plot(x, baseline, label="baseline")
# plt.xlim(2550, 2600)

# 1. -----------------
# plt.xlim(2700, 3400)
plt.ylim(-0.010, 0.0045)
plt.ylim(-0.01, 0.01)
plt.xlim(500,4000)
#----------------------

# plt.xlim(2700,)

plt.plot(x, np.zeros(len(y)), label="y = 0",color='black')
# plt.gca().invert_xaxis()
plt.xlabel("Wavenumber (cm⁻¹)")
plt.ylabel("Absorbance")
plt.legend(); plt.title("Roy et al. Hierarchical Baseline Correction")
plt.show()


In [ ]:
# ======================================
# Replicate Roy & Maiti dual-axis style
# ======================================

i = 4
normVP = [420, 420, 428, 448, 417, 430, 420, 449, 483, 499, 
    438, 465, 438, 428, 503, 505, 504, 454, 515, 441, 
    404, 363]
filename, df = dataframes[i]
norm_factor_i = normVP[i]
x = df['Wavenumber'].values
y = df['Intensity'].values

# segment_edges = [990, 1020]
segment_edges = [500, 4000]
y_stage3, y_stage2, y_stage1 = baseline_roy(x, y, norm_factor_i=norm_factor_i)
# y_stage2, y_stage1 = baseline_intermediates(x, y, norm_factor_i = norm_factor_i)
baseline = y - y_stage3
fig, ax1 = plt.subplots(figsize=(14,6))

# Primary plot (left axis)
ax1.plot(x, y, label="Original", color='royalblue', linewidth=1.6)
# ax1.plot(x, y_stage2, label="2nd order", color='lightgrey', linewidth=2)
# ax1.plot(x, y_stage3, label="3rd order", color='firebrick', linewidth=2)

# Zero reference lines (left + right)
ax1.axhline(0, linestyle='--', color='royalblue', linewidth=1)
# ax2 line added later

# Axis labels & scale formatting
ax1.set_xlabel("Wavenumbers in cm$^{-1}$", fontsize=14)
ax1.set_ylabel(r"Absorbance ($\times \,10^{-3}$)", fontsize=14, color='royalblue')
ax1.tick_params(axis='y', labelcolor='royalblue', labelsize=12)

# ---------------
ax1.set_ylim(-0.008, 0.002)         # adjust as needed
ax1.set_xlim(2135, 2200)            
# -------------------

# ----------------------
# ax1.set_ylim(-0.006, 0.004)
# ax1.set_xlim(1150, 1400)
# ------------------------
# -------------------------------------------------------------------
# Second axis (right)
ax2 = ax1.twinx()

# Re-plot corrected data scaled differently if desired
ax2.plot(x, y_stage2, label="2nd order", color='grey', linewidth=2)
ax2.plot(x, y_stage3, label="3rd order", color='firebrick', linewidth=2)
ax2.plot(x, y_stage3*1000, color='firebrick', alpha=0)  # invisible, only sets scale
ax2.set_ylabel(r"Absorbance ($\times \,10^{-3}$)", fontsize=14, color='darkorange')
ax2.tick_params(axis='y', labelcolor='darkorange', labelsize=12)
# --------------------------------
# ax2.set_ylim(-0.0025, 0.012)
# ---------------------------------
ax2.set_ylim(-0.005, 0.004)
# --------------------------

                 # second y-axis range
ax2.axhline(0, linestyle='--', color='darkorange', linewidth=1)

# -------------------------------------------------------------------
# Collect handles from both axes
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(lines1 + lines2, labels1 + labels2,
           loc="upper left", fontsize=13)


plt.title("Roy & Maiti Hierarchical Baseline Correction", fontsize=16)
plt.tight_layout()
plt.show()


### Category Wise Mean Spectra plots 

In [ ]:
df = df_whole                                                                                                                                                                                      # ── OPTIONS ───────────────────────────────────────────────────                                                                                                                                
cancer_group = ['PC', 'KC', 'BC']           # change to ['PC', 'KC', 'BC'] for all cancer
x_range = (990,1020)                  # e.g. (900, 1800) or None for full range                                                                                                                       
y_range = (-0.0002,0.0002)                  # e.g. (-0.005, 0.02) or None for auto                                                                                                                          
# ──────────────────────────────────────────────────────────────                                                                                                                                
                                                                                                                                                                                                
H_spectra      = np.array(df[df['class'] == 'H']['intensity_baseline_corrected'].tolist())                                                                                                      
cancer_spectra = np.array(df[df['class'].isin(cancer_group)]['intensity_baseline_corrected'].tolist())
                                                                                                                                                                                                
H_mean, H_std           = H_spectra.mean(axis=0),      H_spectra.std(axis=0)                                                                                                                    
cancer_mean, cancer_std = cancer_spectra.mean(axis=0),  cancer_spectra.std(axis=0)                                                                                                              
                                                                                                                                                                                                
cancer_label = 'KC + BC + PC'
                                                                                                                                                                                                
fig, ax = plt.subplots(figsize=(8, 5))                                                                                                                                                         

ax.plot(x, H_mean,      color='blue', label=f'H (n={len(H_spectra)})')                                                                                                                          
ax.fill_between(x, H_mean - H_std, H_mean + H_std, color='lightblue', alpha=0.2)
                                                                                                                                                                                                
ax.plot(x, cancer_mean, color='red',  label=f'{cancer_label} (n={len(cancer_spectra)})')                                                                                                        
ax.fill_between(x, cancer_mean - cancer_std, cancer_mean + cancer_std, color='pink', alpha=0.2)                                                                                                  
                                                                                                                                                                                                
if x_range: ax.set_xlim(x_range)                                                                                                                                                                
if y_range: ax.set_ylim(y_range)
                                                                                                                                                                                                
# ax.invert_xaxis()
ax.set_xlabel('Wavenumber (cm⁻¹)')
ax.set_ylabel('Intensity (baseline corrected)')                                                                                                                                                
ax.set_title(f'Mean ± 1 SD: H vs {cancer_label}')
ax.legend()                                                                                                                                                                                     
plt.tight_layout()
plt.show()                                                                                                                                                                                      
                                                                                                                                                                                                


In [ ]:
df.info

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

labels = np.array([
    ['$\\mathbf{True Positive}$\n$Y=+1,\\ \\hat{Y}=+1$', '$\\mathbf{False Negative}$\n$Y=+1,\\ \\hat{Y}=-1$'],
    ['$\\mathbf{False Positive}$\n$Y=-1,\\ \\hat{Y}=+1$', '$\\mathbf{True Negative}$\n$Y=-1,\\ \\hat{Y}=-1$']
])

color_matrix = np.array([[1, 0], 
                         [0, 1]])

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(color_matrix, annot=labels, fmt='', cmap=["#FFD6E0", "#FFFFFF"], cbar=False,
            linewidths=2, linecolor='black', annot_kws={"size": 15},
            xticklabels=['Cancer $(+1)$', 'Healthy $(-1)$'],
            yticklabels=['Cancer $(+1)$', 'Healthy $(-1)$'])

ax.set_xlabel('Predicted Class $(\\hat{Y})$', fontsize=15, weight='bold')
ax.set_ylabel('True Class $(Y)$', fontsize=15, weight='bold')
plt.title('Confusion Matrix Outcomes', fontsize=15, pad=10)

plt.tight_layout()
plt.savefig('confusion_matrix.pdf', format='pdf', bbox_inches='tight')
plt.show()